In [1]:
%pip install rdflib owlready2 -q

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install graphviz pydotplus -q

  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> [104 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build
      creating build/lib
      creating build/lib/pydotplus
      copying lib/pydotplus/__init__.py -> build/lib/pydotplus
      copying lib/pydotplus/graphviz.py -> build/lib/pydotplus
      copying lib/pydotplus/parser.py -> build/lib/pydotplus
      copying lib/pydotplus/version.py -> build/lib/pydotplus
      /usr/lib/python3/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
      !!
      
              ********************************************************************************
              Please avoid running ``setup.py`` directly.
              Instead, use pypa/build, pypa/installer or other
              standards-based tools.
      
              See https://blog.ganssle.io/article

ERROR: Could not build wheels for pydotplus, which is required to install pyproject.toml-based projects


Note: you may need to restart the kernel to use updated packages.


In [3]:
# Java нужна только для встроенного reasoner'а (sync_reasoner). Аксиомы заданы на Python,
# поэтому установка не обязательна:
# !apt-get install -y openjdk-11-jdk -qq

In [4]:
import pandas as pd
import os
from owlready2 import *

## Загрузка данных

In [5]:
# Загружаем данные из CSV файлов
# Папка с данными ищется автоматически (в Colab и локально)
DATA_DIR = next(p for p in ['ontology_data', 'ontology_data/ontology_data', '../ontology_data/ontology_data']
                if os.path.exists(os.path.join(p, 'knowledge.csv')))

def load_data():
    knowledge_df = pd.read_csv(f'{DATA_DIR}/knowledge.csv')
    competencies_df = pd.read_csv(f'{DATA_DIR}/competencies.csv')
    competency_knowledge_df = pd.read_csv(f'{DATA_DIR}/competency_knowledge.csv')
    positions_df = pd.read_csv(f'{DATA_DIR}/positions.csv')
    position_competency_df = pd.read_csv(f'{DATA_DIR}/position_competency.csv')
    employees_df = pd.read_csv(f'{DATA_DIR}/employees.csv')

    return knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df

In [6]:
knowledge_df, competencies_df, competency_knowledge_df, positions_df, position_competency_df, employees_df = load_data()

In [7]:
knowledge_df

,knowledge_id,knowledge_name
0,1,Математическая статистика
1,2,Линейная алгебра
2,3,Теория вероятностей
3,4,Python программирование
4,5,"Библиотеки ML (scikit-learn, TensorFlow, PyTorch)"
5,6,Обработка естественного языка (NLP)
6,7,Компьютерное зрение
7,8,"Работа с большими данными (Spark, Hadoop)"
8,9,SQL и базы данных
9,10,Визуализация данных


# Создание новой онтологии

## Итерация 1.
Создадим знания и компетенции

In [8]:
# Создаем онтологию
onto = get_ontology("http://test.org/ml_specialist_v1.owl")

with onto:
    class Знание(Thing):
        pass

    class Компетенция(Thing):
        pass

    class требует_знание(ObjectProperty):
        domain = [Компетенция]
        range = [Знание]

    class название(DataProperty):
        domain = [Thing]
        range = [str]

### Создание индивидуальностей для знаний и компетенций



In [9]:
def create_knowledge(knowledge_df):
  with onto:

    knowledge_instances = {}
    for _, row in knowledge_df.iterrows():
        knowledge = Знание(f"знание_{row['knowledge_id']}")
        knowledge.название = [row['knowledge_name']]
        knowledge_instances[row['knowledge_id']] = knowledge
    return knowledge_instances

def create_competencies( competencies_df):
  with onto:
    competency_instances = {}
    for _, row in competencies_df.iterrows():
        competency = Компетенция(f"компетенция_{row['competency_id']}")
        competency.название = [row['competency_name']]
        competency_instances[row['competency_id']] = competency
    return competency_instances


In [10]:
knowledges = create_knowledge(knowledge_df)

In [11]:
knowledges

{1: ml_specialist_v1.знание_1,
 2: ml_specialist_v1.знание_2,
 3: ml_specialist_v1.знание_3,
 4: ml_specialist_v1.знание_4,
 5: ml_specialist_v1.знание_5,
 6: ml_specialist_v1.знание_6,
 7: ml_specialist_v1.знание_7,
 8: ml_specialist_v1.знание_8,
 9: ml_specialist_v1.знание_9,
 10: ml_specialist_v1.знание_10}

In [12]:
for id, item in knowledges.items():
  print (item.название[0])

Математическая статистика
Линейная алгебра
Теория вероятностей
Python программирование
Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Обработка естественного языка (NLP)
Компьютерное зрение
Работа с большими данными (Spark, Hadoop)
SQL и базы данных
Визуализация данных


In [13]:
competencies = create_competencies( competencies_df)
for id, item in competencies.items():
  print (item.название[0])

Математическая подготовка
Программирование и алгоритмы
Машинное обучение
Глубокое обучение
Обработка и анализ данных
Работа с большими данными
Визуализация и представление данных
Развертывание ML-моделей


### Создание отношений между знаниями и компетенциями

In [14]:
# Связываем компетенции со знаниями
def  relate_competencies_knowledge (competency_knowledge_df,  competency_instances, knowledge_instances):
  with onto:
    for _, row in competency_knowledge_df.iterrows():
        competency = competency_instances[row['competency_id']]
        knowledge = knowledge_instances[row['knowledge_id']]
        competency.требует_знание.append(knowledge)


In [15]:
relate_competencies_knowledge (competency_knowledge_df, competencies, knowledges)

Выведем связи между компенциями и знаниями

In [16]:
def print_competency_knowledge():
  print("\n" + "="*60)
  print("СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ")
  print("="*60)

  for competency_id, competency in competencies.items():
      comp_name = competency.название[0]
      print(f"\n{comp_name} требует знания:")

      required_knowledge = list(competency.требует_знание)
      for know in required_knowledge:
          print(f"  - {know.название[0]}")

print_competency_knowledge()



СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

# Итерация 2

Добавление новых концептов в онтологию


In [17]:
with onto:
  class Специалист(Thing):
        pass

  class Должность(Thing):
        pass

  class обладает_знанием(ObjectProperty):
        domain = [Специалист]
        range = [Знание]

  class обладает_компетенцией(ObjectProperty):
        domain = [Специалист]
        range = [Компетенция]

  class может_занимать_должность(ObjectProperty):
        domain = [Специалист]
        range = [Должность]

  class требует_компетенцию(ObjectProperty):
        domain = [Должность]
        range = [Компетенция]

  class имя_сотрудника(DataProperty):
        domain = [Специалист]
        range = [str]

In [18]:
employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'].apply(list)

employee_id  full_name                     
1            Иванов Алексей Сергеевич           [1, 4, 5]
2            Петрова Мария Владимировна         [1, 4, 5]
3            Сидоров Дмитрий Петрович           [1, 2, 9]
4            Козлова Анна Игоревна              [5, 6, 7]
5            Федоров Максим Андреевич           [5, 6, 7]
6            Николаева Екатерина Дмитриевна    [4, 5, 10]
7            Орлов Сергей Викторович            [1, 2, 3]
Name: knowledge_id, dtype: object

### Создание индивидуальностей для сотрудников и их отношений с знаниями


In [19]:
def create_employees(employees_df, knowledge_instances):
  employees_df = employees_df.drop_duplicates()
  employee_knowledge = employees_df.groupby(['employee_id', 'full_name'])['knowledge_id'] \
  .apply(lambda x: list(set(x))).reset_index()

  employee_instances = {}
  for _, row in employee_knowledge.iterrows():
    # создадим сотрудника
      employee = Специалист(f"сотрудник_{row['employee_id']}")
      employee.имя_сотрудника = [row['full_name']]
      employee_instances[row['employee_id']] = employee

      # добавим знания сотруднику
      for knowledge_id in row['knowledge_id']:
          if knowledge_id in knowledge_instances:
              employee.обладает_знанием.append(knowledge_instances[knowledge_id])

  return employee_instances

In [20]:
employees = create_employees(employees_df, knowledges)

In [21]:
def print_employee_mindmap():
  print("\n" + "="*60)
  print("КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА")
  print("="*60)
  for employee_id, employee in employees.items():
          employee_name = employee.имя_сотрудника[0] if employee.имя_сотрудника else f"Сотрудник {employee_id}"

          print(f"\n{employee_name}:")

          # Получаем список знаний сотрудника
          knowledge_list = list(employee.обладает_знанием)

          if knowledge_list:
              print("Знания:")
              for knowledge in knowledge_list:
                  know_name = knowledge.название[0] if knowledge.название else knowledge.name
                  print(f"  - {know_name}")
          else:
              print("Знания: нет")

print_employee_mindmap()


КАРТА ЗНАНИЙ КАЖДОГО СОТРУДНИКА

Иванов Алексей Сергеевич:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Петрова Мария Владимировна:
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Сидоров Дмитрий Петрович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных

Козлова Анна Игоревна:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Федоров Максим Андреевич:
Знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Николаева Екатерина Дмитриевна:
Знания:
  - Визуализация данных
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Орлов Сергей Викторович:
Знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей


### Создание индивидуальностей для должностей и из связей с компетенциями


In [22]:
# Создаем экземпляры должностей
def create_position_instances(positions_df):
    position_instances = {}
    for _, row in positions_df.iterrows():
        position = Должность(f"должность_{row['position_id']}")
        position.name = row['position_name']
        position_instances[row['position_id']] = position
    return position_instances

In [23]:
positions = create_position_instances(positions_df)
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [24]:
# Устанавливаем связи между должностями и компетенциями
def set_position_competency_relations(position_competency_df, position_instances, competency_instances):
    for _, row in position_competency_df.iterrows():
        position = position_instances[row['position_id']]
        competency = competency_instances[row['competency_id']]
        position.требует_компетенцию.append(competency)

In [25]:
set_position_competency_relations(position_competency_df, positions, competencies)

In [26]:
# сюда вставить код для выполнения самостоятельных заданий
# (задания 1-4 выполнены ниже, каждое - в НОВОЙ версии онтологии)

## Выведем требования к должностям

In [27]:
# Проверим требования должностей
def print_position_require():
  print("\n" + "="*60)
  print("ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ")
  print("="*60)

  for position_id, position in positions.items():
      pos_name = position.name
      print(f"\n{pos_name} требует компетенции:")

      required_competencies = list(position.требует_компетенцию)
      if required_competencies:
          for comp in required_competencies:
              print(f"  - {comp.название[0]}")
      else:
          print("  - нет требований")

print_position_require()


ТРЕБОВАНИЯ ДОЛЖНОСТЕЙ

ML Engineer требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Scientist требует компетенции:
  - Программирование и алгоритмы
  - Машинное обучение

Data Analyst требует компетенции:
  - Обработка и анализ данных

NLP Engineer требует компетенции:
  - Программирование и алгоритмы

Computer Vision Engineer требует компетенции:
  - Машинное обучение


## Создание аксиом для онтологии

 <b>Аксиома 1:</b> Если специалист обладает всеми требуемыми знаниями для компетенции, то у него есть эта компетенция

 <b>Аксиома 2:</b> Если у специалиста есть требуемые компетенции для должности, то он может занимать эту должность

  <b> Аксиома 3:</b> (для проверки корректности онтологии) Если специалист занимает должность, то у него должны быть соответствующие компетенции
      


In [28]:
def define_rules():
    # Аксиома 1:
    with onto:
        class ИмеетКомпетенциюПоЗнаниям(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой компетенции проверяем, есть ли у специалиста все необходимые знания
                for competency in onto.Компетенция.instances():
                    required_knowledge = list(competency.требует_знание)
                    specialist_knowledge = list(specialist.обладает_знанием)

                    # Проверяем, что все требуемые знания есть у специалиста
                    if all(knowledge in specialist_knowledge for knowledge in required_knowledge):
                        if competency not in specialist.обладает_компетенцией:
                            specialist.обладает_компетенцией.append(competency)

                return True

        # Аксиома 2:
        class МожетЗаниматьДолжностьПоКомпетенциям(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                # Для каждой должности проверяем, есть ли у специалиста все необходимые компетенции
                for position in onto.Должность.instances():
                    required_competencies = list(position.требует_компетенцию)
                    specialist_competencies = list(specialist.обладает_компетенцией)

                    # Проверяем, что все требуемые компетенции есть у специалиста
                    if all(competency in specialist_competencies for competency in required_competencies):
                        if position not in specialist.может_занимать_должность:
                            specialist.может_занимать_должность.append(position)

                return True

        # Аксиома 3:
        class ПроверитьКомпетенцииДляДолжности(Специалист >> bool):
            def __init__(self):
                super().__init__()

            def __call__(self, specialist):
                positions = list(specialist.может_занимать_должность)
                competencies = list(specialist.обладает_компетенцией)

                for position in positions:
                    required_competencies = list(position.требует_компетенцию)
                    if not all(comp in competencies for comp in required_competencies):
                        print(f"Предупреждение: у {specialist.name} нет всех компетенций для {position.name}")

                return True

    return ИмеетКомпетенциюПоЗнаниям(), МожетЗаниматьДолжностьПоКомпетенциям(), ПроверитьКомпетенцииДляДолжности()

In [29]:
rule1, rule2, rule3 = define_rules()

In [30]:
# Проверим связи знаний и компетенций
print_competency_knowledge()


СВЯЗИ ЗНАНИЙ И КОМПЕТЕНЦИЙ

Математическая подготовка требует знания:
  - Математическая статистика
  - Линейная алгебра
  - Теория вероятностей

Программирование и алгоритмы требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Машинное обучение требует знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)

Глубокое обучение требует знания:
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
  - Обработка естественного языка (NLP)
  - Компьютерное зрение

Обработка и анализ данных требует знания:
  - Математическая статистика
  - SQL и базы данных

Работа с большими данными требует знания:
  - Работа с большими данными (Spark, Hadoop)
  - SQL и базы данных

Визуализация и представление данных требует знания:
  - Визуализация данных
  - Python программирование

Развертывание ML-моделей требует знания:
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, 

In [31]:
onto.Специалист.instances()

[ml_specialist_v1.сотрудник_1,
 ml_specialist_v1.сотрудник_2,
 ml_specialist_v1.сотрудник_3,
 ml_specialist_v1.сотрудник_4,
 ml_specialist_v1.сотрудник_5,
 ml_specialist_v1.сотрудник_6,
 ml_specialist_v1.сотрудник_7]

In [32]:
for employee in onto.Специалист.instances():
        rule1(employee)  # Применяем правило для компетенций
        rule2(employee)  # Применяем правило для должностей
        rule3(employee)  # Проверяем соответствие

## Проведение анализа данных

In [33]:
def print_position_for_employee():
  print("\n" + "="*60)
  print("АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ")
  print("="*60)

  for employee in onto.Специалист.instances():
          print(f"\nСотрудник: {employee.имя_сотрудника[0]}")

          print("Знания:")
          for knowledge in employee.обладает_знанием:
              print(f"  - {knowledge.название[0]}")

          print("Компетенции:")
          for competency in employee.обладает_компетенцией:
              print(f"  - {competency.название[0]}")

          print("Может занимать должности:")
          for position in employee.может_занимать_должность:
              print(f"  - {position.name}")

          print("-" * 30)

print_position_for_employee()


АНАЛИЗ ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ У СОТРУДНИКОВ

Сотрудник: Иванов Алексей Сергеевич
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Петрова Мария Владимировна
Знания:
  - Математическая статистика
  - Python программирование
  - Библиотеки ML (scikit-learn, TensorFlow, PyTorch)
Компетенции:
  - Программирование и алгоритмы
  - Машинное обучение
  - Развертывание ML-моделей
Может занимать должности:
  - ML Engineer
  - Data Scientist
  - NLP Engineer
  - Computer Vision Engineer
------------------------------

Сотрудник: Сидоров Дмитрий Петрович
Знания:
  - Математическая статистика
  - Линейная алгебра
  - SQL и базы данных
Компетенции:
  - Обработка и анализ данных
Может зани

In [34]:
# Убедимся, что у сотрудников достаточно знаний для получения компетенций
def check_employee_competencies(employee_instances, competency_instances):
    """Проверяем, какие компетенции могут получить сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for competency_id, competency in competency_instances.items():
            comp_name = competency.название[0]
            required_knowledge = set(competency.требует_знание)
            employee_knowledge = set(employee.обладает_знанием)

            # Проверяем, есть ли у сотрудника все необходимые знания
            if required_knowledge.issubset(employee_knowledge):
                print(f"  Может получить компетенцию: {comp_name}")
                # Присваиваем компетенцию
                if competency not in employee.обладает_компетенцией:
                    employee.обладает_компетенцией.append(competency)
            else:
                missing_knowledge = required_knowledge - employee_knowledge
                if missing_knowledge:
                    print(f"  Не хватает для '{comp_name}':")
                    for know in missing_knowledge:
                        print(f"      - {know.название[0]}")


In [35]:
check_employee_competencies(employees, competencies)


ПРОВЕРКА ВОЗМОЖНЫХ КОМПЕТЕНЦИЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
  Не хватает для 'Математическая подготовка':
      - Линейная алгебра
      - Теория вероятностей
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      - Обработка естественного языка (NLP)
      - Компьютерное зрение
  Не хватает для 'Обработка и анализ данных':
      - SQL и базы данных
  Не хватает для 'Работа с большими данными':
      - SQL и базы данных
      - Работа с большими данными (Spark, Hadoop)
  Не хватает для 'Визуализация и представление данных':
      - Визуализация данных
  Может получить компетенцию: Развертывание ML-моделей

Петрова Мария Владимировна:
  Не хватает для 'Математическая подготовка':
      - Линейная алгебра
      - Теория вероятностей
  Может получить компетенцию: Программирование и алгоритмы
  Может получить компетенцию: Машинное обучение
  Не хватает для 'Глубокое обучение':
      -

In [36]:
def check_employee_positions(employee_instances, position_instances):
    """Проверяем, какие должности могут занять сотрудники"""
    print("\n" + "="*60)
    print("ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ")
    print("="*60)

    for employee_id, employee in employee_instances.items():
        employee_name = employee.имя_сотрудника[0]
        print(f"\n{employee_name}:")

        for position_id, position in position_instances.items():
            pos_name = position.name
            required_competencies = set(position.требует_компетенцию)
            employee_competencies = set(employee.обладает_компетенцией)

            # Проверяем, есть ли у сотрудника все необходимые компетенции
            if required_competencies.issubset(employee_competencies):
                print(f"   Может занять должность: {pos_name}")
                # Присваиваем должность
                if position not in employee.может_занимать_должность:
                    employee.может_занимать_должность.append(position)
            else:
                missing_competencies = required_competencies - employee_competencies
                if missing_competencies:
                    print(f"   Не хватает для '{pos_name}':")
                    for comp in missing_competencies:
                        print(f"      - {comp.название[0]}")

# Применяем проверки

check_employee_positions(employees, positions)



ПРОВЕРКА ВОЗМОЖНЫХ ДОЛЖНОСТЕЙ СОТРУДНИКОВ

Иванов Алексей Сергеевич:
   Может занять должность: ML Engineer
   Может занять должность: Data Scientist
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Может занять должность: NLP Engineer
   Может занять должность: Computer Vision Engineer

Петрова Мария Владимировна:
   Может занять должность: ML Engineer
   Может занять должность: Data Scientist
   Не хватает для 'Data Analyst':
      - Обработка и анализ данных
   Может занять должность: NLP Engineer
   Может занять должность: Computer Vision Engineer

Сидоров Дмитрий Петрович:
   Не хватает для 'ML Engineer':
      - Программирование и алгоритмы
      - Машинное обучение
   Не хватает для 'Data Scientist':
      - Программирование и алгоритмы
      - Машинное обучение
   Может занять должность: Data Analyst
   Не хватает для 'NLP Engineer':
      - Программирование и алгоритмы
   Не хватает для 'Computer Vision Engineer':
      - Машинное обучение

Козлова Анна 

In [37]:
# Сохраняем исходную онтологию (версия 1) в файл
onto.save(file="ml_specialist_v1.owl", format="rdfxml")
print("Онтология v1 сохранена:", onto.base_iri)

Онтология v1 сохранена: http://test.org/ml_specialist_v1.owl#


# Задание 1:  Добавление необходимой компетенции в должность

Добавить требование, что компетениция NLP Enginner содержит знание по глубокому обучению.

<B>Важно, что добавление в онтологию новых данных в Питоне должно сопровождаться созданием новой онтологии.</B>


In [38]:
positions

{1: ml_specialist_v1.ML Engineer,
 2: ml_specialist_v1.Data Scientist,
 3: ml_specialist_v1.Data Analyst,
 4: ml_specialist_v1.NLP Engineer,
 5: ml_specialist_v1.Computer Vision Engineer}

In [39]:
positions[4].требует_компетенцию

[ml_specialist_v1.компетенция_2]

In [40]:
competencies[2].название

['Программирование и алгоритмы']

In [41]:
for i, item in competencies.items():
  print (i, ' ', item.название)


1   ['Математическая подготовка']
2   ['Программирование и алгоритмы']
3   ['Машинное обучение']
4   ['Глубокое обучение']
5   ['Обработка и анализ данных']
6   ['Работа с большими данными']
7   ['Визуализация и представление данных']
8   ['Развертывание ML-моделей']


In [42]:
# Требование добавляется не в исходную онтологию v1, а в новую онтологию v2 (см. ниже)

Сначала вспомогательные функции: каждая версия онтологии создается заново (новый IRI) и сохраняется в файл.

In [43]:
# ---------------------------------------------------------------
# Вспомогательные функции для заданий 1-4.
# Правило: любое добавление данных = НОВАЯ онтология (новый IRI / новая версия).
# build_ontology заново создает схему (классы, свойства), индивидов и связи,
# применяет новые данные из задания, запускает аксиомы 1-2 и сохраняет файл .owl.
# ---------------------------------------------------------------
def build_ontology(version, modifications=()):
    global onto, Знание, Компетенция, Специалист, Должность
    global knowledges, competencies, employees, positions, rule1, rule2, rule3

    onto = get_ontology(f"http://test.org/ml_specialist_{version}.owl")   # новая онтология
    with onto:
        class Знание(Thing): pass
        class Компетенция(Thing): pass
        class Специалист(Thing): pass
        class Должность(Thing): pass
        class требует_знание(ObjectProperty):
            domain = [Компетенция]; range = [Знание]
        class название(DataProperty):
            domain = [Thing]; range = [str]
        class обладает_знанием(ObjectProperty):
            domain = [Специалист]; range = [Знание]
        class обладает_компетенцией(ObjectProperty):
            domain = [Специалист]; range = [Компетенция]
        class может_занимать_должность(ObjectProperty):
            domain = [Специалист]; range = [Должность]
        class требует_компетенцию(ObjectProperty):
            domain = [Должность]; range = [Компетенция]
        class имя_сотрудника(DataProperty):
            domain = [Специалист]; range = [str]

    kd, cd, ckd, pd_, pcd, ed = load_data()
    knowledges   = create_knowledge(kd)
    competencies = create_competencies(cd)
    relate_competencies_knowledge(ckd, competencies, knowledges)
    employees    = create_employees(ed, knowledges)
    positions    = create_position_instances(pd_)
    set_position_competency_relations(pcd, positions, competencies)

    with onto:
        for mod in modifications:      # новые данные по заданиям
            mod()

    rule1, rule2, rule3 = define_rules()
    for e in onto.Специалист.instances():
        rule1(e); rule2(e); rule3(e)
    onto.save(file=f"ml_specialist_{version}.owl", format="rdfxml")
    print(f"Создана онтология {onto.base_iri} -> файл ml_specialist_{version}.owl")
    return onto

def show_employee(part_of_name):
    """Знания, компетенции и возможные должности сотрудника."""
    for e in onto.Специалист.instances():
        if part_of_name in e.имя_сотрудника[0]:
            print(f"Сотрудник: {e.имя_сотрудника[0]}")
            print("  Знания:      ", [k.название[0] for k in e.обладает_знанием])
            print("  Компетенции: ", [c.название[0] for c in e.обладает_компетенцией])
            print("  Должности:   ", [p.name for p in e.может_занимать_должность])

def position_requirements(position_name):
    pos = next(p for p in onto.Должность.instances() if p.name == position_name)
    print(f"{position_name} требует:", [c.название[0] for c in pos.требует_компетенцию])

def who_can_be(position_name):
    return [e.имя_сотрудника[0] for e in onto.Специалист.instances()
            if any(p.name == position_name for p in e.может_занимать_должность)]

In [44]:
# Функции с новыми данными для заданий 1-4 (вызываются внутри build_ontology)
def mod_task1():
    """Задание 1: NLP Engineer требует компетенцию 'Глубокое обучение'."""
    nlp = next(p for p in positions.values() if p.name == "NLP Engineer")
    dl  = next(c for c in competencies.values() if c.название[0] == "Глубокое обучение")
    if dl not in nlp.требует_компетенцию:
        nlp.требует_компетенцию.append(dl)

def mod_task2():
    """Задание 2: Data Scientist требует компетенцию 'Математическая подготовка'."""
    ds   = next(p for p in positions.values() if p.name == "Data Scientist")
    math = next(c for c in competencies.values() if c.название[0] == "Математическая подготовка")
    if math not in ds.требует_компетенцию:
        ds.требует_компетенцию.append(math)

def mod_task3():
    """Задание 3: Федорову добавляем знания, которых не хватает для Computer Vision Engineer."""
    fedorov = next(e for e in employees.values() if "Федоров Максим" in e.имя_сотрудника[0])
    cv = next(p for p in positions.values() if p.name == "Computer Vision Engineer")
    for comp in cv.требует_компетенцию:
        for k in comp.требует_знание:
            if k not in fedorov.обладает_знанием:
                fedorov.обладает_знанием.append(k)
                print(f"  + Федорову добавлено знание: {k.название[0]}")

def mod_task4():
    """Задание 4: новый сотрудник (мое ФИО) со знаниями для ВСЕХ должностей онтологии."""
    me = Специалист("сотрудник_8")
    me.имя_сотрудника = ["Чикишев Степан Максимович"]
    employees[8] = me
    for pos in positions.values():
        for comp in pos.требует_компетенцию:
            for k in comp.требует_знание:
                if k not in me.обладает_знанием:
                    me.обладает_знанием.append(k)

### Результат задания 1: онтология v2

In [45]:
# Версия 2: исходные данные + задание 1
onto_v2 = build_ontology("v2", [mod_task1])
position_requirements("NLP Engineer")
print()
print("Кто может быть NLP Engineer:", who_can_be("NLP Engineer"))

Создана онтология http://test.org/ml_specialist_v2.owl# -> файл ml_specialist_v2.owl
NLP Engineer требует: ['Программирование и алгоритмы', 'Глубокое обучение']

Кто может быть NLP Engineer: []


###

## Задание 2:
Добавьте требование в онтологию: должность Data Scientist требует компетенции "Математическая подготовка"

In [46]:
# Версия 3: задание 1 + задание 2
onto_v3 = build_ontology("v3", [mod_task1, mod_task2])
position_requirements("Data Scientist")
print()
print("Кто может быть Data Scientist:", who_can_be("Data Scientist"))

Создана онтология http://test.org/ml_specialist_v3.owl# -> файл ml_specialist_v3.owl
Data Scientist требует: ['Программирование и алгоритмы', 'Машинное обучение', 'Математическая подготовка']

Кто может быть Data Scientist: []


## Задание 3:
Добавьте сотруднику Федорову Максиму Андреевичу необходимые знания, чтобы он мог претендовать на должность Computer Vision Engineer

In [47]:
# Проверка ДО: Федоров в версии 3 не может быть Computer Vision Engineer
print("Версия 3 (до добавления знаний):")
show_employee("Федоров")
print("CV Engineer:", who_can_be("Computer Vision Engineer"))
print()
# Версия 4: задания 1-3
onto_v4 = build_ontology("v4", [mod_task1, mod_task2, mod_task3])
print()
print("Версия 4 (после добавления знаний):")
show_employee("Федоров")
print("CV Engineer:", who_can_be("Computer Vision Engineer"))

Версия 3 (до добавления знаний):
Сотрудник: Федоров Максим Андреевич
  Знания:       ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)', 'Обработка естественного языка (NLP)', 'Компьютерное зрение']
  Компетенции:  ['Глубокое обучение']
  Должности:    []
CV Engineer: ['Иванов Алексей Сергеевич', 'Петрова Мария Владимировна']

  + Федорову добавлено знание: Математическая статистика
  + Федорову добавлено знание: Python программирование
Создана онтология http://test.org/ml_specialist_v4.owl# -> файл ml_specialist_v4.owl

Версия 4 (после добавления знаний):
Сотрудник: Федоров Максим Андреевич
  Знания:       ['Библиотеки ML (scikit-learn, TensorFlow, PyTorch)', 'Обработка естественного языка (NLP)', 'Компьютерное зрение', 'Математическая статистика', 'Python программирование']
  Компетенции:  ['Программирование и алгоритмы', 'Машинное обучение', 'Глубокое обучение', 'Развертывание ML-моделей']
  Должности:    ['ML Engineer', 'NLP Engineer', 'Computer Vision Engineer']
CV Engineer: ['И

Федоров знал NLP, компьютерное зрение и ML-библиотеки (знания 5, 6, 7), но ему не хватало математической статистики и Python (компетенция «Машинное обучение»). Они добавлены в онтологию v4.

## Задание 4:
Добавьте требование в онтологию нового сотрудника, укажите в качестве имя_сотрудника ваше ФИО. Добавьте знания для данного сотрудника, чтобы он смог претендовать на все должности текущей онтологии

In [48]:
# Версия 5: задания 1-4
onto_v5 = build_ontology("v5", [mod_task1, mod_task2, mod_task3, mod_task4])
print()
show_employee("Чикишев")
print()
all_pos = sorted(p.name for p in onto.Должность.instances())
mine = sorted(p.name for e in onto.Специалист.instances() if "Чикишев" in e.имя_сотрудника[0] for p in e.может_занимать_должность)
print("Все должности онтологии:", all_pos)
print("Мои должности:          ", mine)
print("Может занимать ВСЕ должности:", all_pos == mine)

  + Федорову добавлено знание: Математическая статистика
  + Федорову добавлено знание: Python программирование


Создана онтология http://test.org/ml_specialist_v5.owl# -> файл ml_specialist_v5.owl

Сотрудник: Чикишев Степан Максимович
  Знания:       ['Python программирование', 'Библиотеки ML (scikit-learn, TensorFlow, PyTorch)', 'Математическая статистика', 'Линейная алгебра', 'Теория вероятностей', 'SQL и базы данных', 'Обработка естественного языка (NLP)', 'Компьютерное зрение']
  Компетенции:  ['Математическая подготовка', 'Программирование и алгоритмы', 'Машинное обучение', 'Глубокое обучение', 'Обработка и анализ данных', 'Развертывание ML-моделей']
  Должности:    ['ML Engineer', 'Data Scientist', 'Data Analyst', 'NLP Engineer', 'Computer Vision Engineer']

Все должности онтологии: ['Computer Vision Engineer', 'Data Analyst', 'Data Scientist', 'ML Engineer', 'NLP Engineer']
Мои должности:           ['Computer Vision Engineer', 'Data Analyst', 'Data Scientist', 'ML Engineer', 'NLP Engineer']
Может занимать ВСЕ должности: True


In [49]:
# Итоговая таблица: кто на какие должности может претендовать в финальной онтологии (v5)
import pandas as pd
rows = []
for e in sorted(onto.Специалист.instances(), key=lambda x: x.name):
    rows.append({"Сотрудник": e.имя_сотрудника[0],
                 "Должности": ", ".join(sorted(p.name for p in e.может_занимать_должность)) or "-"})
pd.DataFrame(rows)

,Сотрудник,Должности
0,Иванов Алексей Сергеевич,"Computer Vision Engineer, ML Engineer"
1,Петрова Мария Владимировна,"Computer Vision Engineer, ML Engineer"
2,Сидоров Дмитрий Петрович,Data Analyst
3,Козлова Анна Игоревна,-
4,Федоров Максим Андреевич,"Computer Vision Engineer, ML Engineer, NLP Eng..."
5,Николаева Екатерина Дмитриевна,-
6,Орлов Сергей Викторович,-
7,Чикишев Степан Максимович,"Computer Vision Engineer, Data Analyst, Data S..."


**Вывод.** Финальная онтология v5 содержит все изменения заданий 1-4; нового сотрудника Чикишева С.М. аксиомы 1-2 относят ко всем пяти должностям.